In [3]:
import pandas as pd
import numpy as np
import os
from sklearn.preprocessing import LabelEncoder, StandardScaler
import joblib



In [4]:
dados_caminho = "../dados/dados_nao_processados/"
arquivos_csv = [f for f in os.listdir(dados_caminho) if f.endswith('.csv')]

# carregando todos os arquivos csv do dataset
dataframes = []
for arquivo in arquivos_csv:
    print(f"Carregando: {arquivo}")
    df = pd.read_csv(
        os.path.join(dados_caminho, arquivo),
        encoding='utf-8',
        low_memory=False
    )
    dataframes.append(df)

df_total = pd.concat(dataframes, ignore_index=True)

print(f"\n Dataset carregado")
print(f" Registros: {df_total.shape[0]:,}")
print(f" Colunas:   {df_total.shape[1]}")


Carregando: Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
Carregando: Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
Carregando: Friday-WorkingHours-Morning.pcap_ISCX.csv
Carregando: Monday-WorkingHours.pcap_ISCX.csv
Carregando: Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv
Carregando: Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv
Carregando: Tuesday-WorkingHours.pcap_ISCX.csv
Carregando: Wednesday-workingHours.pcap_ISCX.csv

 Dataset carregado
 Registros: 2,830,743
 Colunas:   79


In [5]:
print("Antes de remover os espaços:")
print(list(df_total.columns[:5]))

# só para remover espaços antes e depois dos nomes das colunas
df_total.columns = df_total.columns.str.strip()

print("\n Depois de remover:")
print(list(df_total.columns[:5]))
print(f"\n Nomes das colunas limpos")


Antes de remover os espaços:
[' Destination Port', ' Flow Duration', ' Total Fwd Packets', ' Total Backward Packets', 'Total Length of Fwd Packets']

 Depois de remover:
['Destination Port', 'Flow Duration', 'Total Fwd Packets', 'Total Backward Packets', 'Total Length of Fwd Packets']

 Nomes das colunas limpos


In [6]:
print("Antes do tratamento:")
infinitos_antes = np.isinf(df_total.select_dtypes(include=[np.number])).sum().sum()
print(f"    Valores infinitos: {infinitos_antes:,}")

# substitui qualquer valor por NaN
df_total = df_total.replace([np.inf, -np.inf], np.nan)

print("\nDepois do tratamento:")
infinitos_depois = np.isinf(df_total.select_dtypes(include=[np.number])).sum().sum()
print(f"    Valores infinitos: {infinitos_depois:,}")
print(f"\n {infinitos_antes:,} valores infinitos convertidos para NaN")


Antes do tratamento:
    Valores infinitos: 4,376

Depois do tratamento:
    Valores infinitos: 0

 4,376 valores infinitos convertidos para NaN


In [7]:
nulos_antes = df_total.isnull().sum().sum()
print(f"Total de valores NaN antes: {nulos_antes:,}")

# removendo linhas com NaN pois são cerca de 0,2% de todos os registros então não fará diferença e nem vale a pena tentar imputar eles
df_total = df_total.dropna()

nulos_depois = df_total.isnull().sum().sum()
print(f"Total de valores NaN depois: {nulos_depois:,}")
print(f"\n Linhas com NaN removidas")
print(f" Registros restantes depois da remoção: {df_total.shape[0]:,}")


Total de valores NaN antes: 5,734
Total de valores NaN depois: 0

 Linhas com NaN removidas
 Registros restantes depois da remoção: 2,827,876


In [8]:
print(f"Registros antes: {df_total.shape[0]:,}")

duplicados = df_total.duplicated().sum()
print(f"Registros duplicados encontrados: {duplicados:,} ({duplicados/len(df_total)*100:.2f}%)")


# removendo os duplicados para realizar um melhor estudo dos dados
df_total = df_total.drop_duplicates()


print(f"\n Registros duplicados removidos")
print(f" Registros restantes depois da remoção: {df_total.shape[0]:,}")



Registros antes: 2,827,876
Registros duplicados encontrados: 307,078 (10.86%)

 Registros duplicados removidos
 Registros restantes depois da remoção: 2,520,798


In [10]:
# mostrando a quantidade de registros de cada classe de forma organizada, com sua porcentagem e o valor total após as remoções anteriores

print("=== DISTRIBUIÇÃO DAS CLASSES (APÓS LIMPEZA) ===\n")
distribuicao = df_total['Label'].value_counts()

for classe, quantidade in distribuicao.items():
    percentual = (quantidade / len(df_total)) * 100
    print(f"  {classe:30s} → {quantidade:>10,} ({percentual:.2f}%)")

print(f"\n Total de registros:  {df_total.shape[0]:,}")
print(f"Total de classes:    {df_total['Label'].nunique()}")


=== DISTRIBUIÇÃO DAS CLASSES (APÓS LIMPEZA) ===

  BENIGN                         →  2,095,057 (83.11%)
  DoS Hulk                       →    172,846 (6.86%)
  DDoS                           →    128,014 (5.08%)
  PortScan                       →     90,694 (3.60%)
  DoS GoldenEye                  →     10,286 (0.41%)
  FTP-Patator                    →      5,931 (0.24%)
  DoS slowloris                  →      5,385 (0.21%)
  DoS Slowhttptest               →      5,228 (0.21%)
  SSH-Patator                    →      3,219 (0.13%)
  Bot                            →      1,948 (0.08%)
  Web Attack � Brute Force       →      1,470 (0.06%)
  Web Attack � XSS               →        652 (0.03%)
  Infiltration                   →         36 (0.00%)
  Web Attack � Sql Injection     →         21 (0.00%)
  Heartbleed                     →         11 (0.00%)

 Total de registros:  2,520,798
Total de classes:    15


In [11]:
# atributos = todas as colunas exceto o rótulo dos registros
# rotulos = a coluna label, ou seja, os tipos de registros
atributos = df_total.drop('Label', axis=1)
rotulos = df_total['Label']

print(f" Formato dos atributos: {atributos.shape}")
print(f"  Formato dos rótulos:   {rotulos.shape}")


 Formato dos atributos: (2520798, 78)
  Formato dos rótulos:   (2520798,)


In [12]:
print("Antes: Os registros estavam sendo tratados pelo label:")
print(rotulos.head(10).tolist())

# criar e aplicar o LabelEncoder para que cada registro tenha um codigo, assim fica mais fácil de manusear 
codificador_rotulos = LabelEncoder()
rotulos_codificados = codificador_rotulos.fit_transform(rotulos)

print("\nDepois: Os registros agora são tratados pelo seu código")
print(rotulos_codificados[:10])

print("\n=== MAPEAMENTO ===")
for i, classe in enumerate(codificador_rotulos.classes_):
    print(f"  {i:2d} → {classe}")



Antes: Os registros estavam sendo tratados pelo label:
['BENIGN', 'BENIGN', 'BENIGN', 'BENIGN', 'BENIGN', 'BENIGN', 'BENIGN', 'BENIGN', 'BENIGN', 'BENIGN']

Depois: Os registros agora são tratados pelo seu código
[0 0 0 0 0 0 0 0 0 0]

=== MAPEAMENTO ===
   0 → BENIGN
   1 → Bot
   2 → DDoS
   3 → DoS GoldenEye
   4 → DoS Hulk
   5 → DoS Slowhttptest
   6 → DoS slowloris
   7 → FTP-Patator
   8 → Heartbleed
   9 → Infiltration
  10 → PortScan
  11 → SSH-Patator
  12 → Web Attack � Brute Force
  13 → Web Attack � Sql Injection
  14 → Web Attack � XSS


In [14]:
print("Antes da normalização:")
print(f"  Min:    {atributos.values.min():.2f}")
print(f"  Max:    {atributos.values.max():,.2f}")
print(f"  Média:  {atributos.values.mean():,.2f}")

# aplicando StandardScaler: cada atributo fica com média 0 e desvio padrão 1
# isso será útil em próximas etapas quando for feita a aplicação de PCA E LDA, e também quando for aplicado o modelo Mean Shift
normalizador = StandardScaler()
atributos_normalizados = normalizador.fit_transform(atributos)

# converte de volta para DataFrame para manter os nomes das colunas
atributos_normalizados = pd.DataFrame(atributos_normalizados, columns=atributos.columns)

print("\nDepois da normalização:")
print(f"  Min:    {atributos_normalizados.values.min():.2f}")
print(f"  Max:    {atributos_normalizados.values.max():.2f}")
print(f"  Média:  {atributos_normalizados.values.mean():.2f}")


Antes da normalização:
  Min:    -32212234632.00
  Max:    2,071,000,000.00
  Média:  1,513,328.25

Depois da normalização:
  Min:    -1443.87
  Max:    1218.27
  Média:  -0.00


In [15]:
# criando pastas de saída para os dados após processamento
os.makedirs("../dados/dados_processados", exist_ok=True)
os.makedirs("../modelos", exist_ok=True)

# salvando os dados processados
atributos_normalizados.to_csv("../dados/dados_processados/atributos_processados.csv", index=False)
pd.Series(rotulos_codificados, name='Label').to_csv("../dados/dados_processados/rotulos_processados.csv", index=False)

# salvando os transformadores (normalizador e codificador) para usar depois em próximas etapas
joblib.dump(normalizador, "../modelos/normalizador.pkl")
joblib.dump(codificador_rotulos, "../modelos/codificador_rotulos.pkl")

print(" Dados processados salvos:")
print("    dados/dados_processados/atributos_processados.csv")
print("    dados/dados_processados/rotulos_processados.csv")
print("\n Transformadores salvos:")
print("    modelos/normalizador.pkl")
print("    modelos/codificador_rotulos.pkl")


 Dados processados salvos:
    dados/dados_processados/atributos_processados.csv
    dados/dados_processados/rotulos_processados.csv

 Transformadores salvos:
    modelos/normalizador.pkl
    modelos/codificador_rotulos.pkl


In [16]:
print("=" * 60)
print("       RESUMO DO PRÉ-PROCESSAMENTO")
print("=" * 60)
print(f"\n Registros iniciais:        2,830,743")
print(f" Registros finais:          {df_total.shape[0]:,}")
print(f" Registros removidos:       {2830743 - df_total.shape[0]:,}")
print(f"\n Atributos :       {atributos_normalizados.shape[1]}")
print(f"  Classes (tipos de registros):          {len(codificador_rotulos.classes_)}")
print(f"\n Dados normalizados:        Sim (StandardScaler)")
print(f" Rótulos codificados:       Sim (LabelEncoder)")
print(f" Dados salvos em:           dados/dados_processados/")
print("=" * 60)


       RESUMO DO PRÉ-PROCESSAMENTO

 Registros iniciais:        2,830,743
 Registros finais:          2,520,798
 Registros removidos:       309,945

 Atributos :       78
  Classes (tipos de registros):          15

 Dados normalizados:        Sim (StandardScaler)
 Rótulos codificados:       Sim (LabelEncoder)
 Dados salvos em:           dados/dados_processados/
